# 🏙️ European Cities — Web Scraping + APIs → MySQL

**Pipeline:**
```
Wikipedia          → cities + populations
OpenWeatherMap API → weathers (current + forecast)
AeroDataBox API    → airports + cities_airports + flights
pandas normalize   → 6 relational tables → MySQL
```

### Schema (6 tabele — exact bootcamp)
```
cities ──┬── populations
         ├── weathers
         └── cities_airports ── airports ── flights
```

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import time
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta

print('Libraries loaded ✅')

## 2. Config — API Keys & DB

⚠️ **Schimbă toate valorile de mai jos înainte să rulezi!**

In [ ]:
OWM_API_KEY      = 'YOUR_OPENWEATHERMAP_KEY'   # OpenWeatherMap
RAPIDAPI_KEY     = 'YOUR_RAPIDAPI_KEY'          # RapidAPI (AeroDataBox)

DB_HOST          = 'localhost'
DB_USER          = 'root'
DB_PASSWORD      = 'YOUR_MYSQL_PASSWORD'
DB_PORT          = 3306
DB_NAME          = 'gans_cities'

print('Config set ✅')

## 3. Define Cities

In [ ]:
CITIES_META = [
    # (city_name,           wiki_slug,             country_code)
    ('Berlin',              'Berlin',               'DE'),
    ('Hamburg',             'Hamburg',              'DE'),
    ('Munich',              'Munich',               'DE'),
    ('Paris',               'Paris',                'FR'),
    ('Lyon',                'Lyon',                 'FR'),
    ('Marseille',           'Marseille',            'FR'),
    ('Rome',                'Rome',                 'IT'),
    ('Milan',               'Milan',                'IT'),
    ('Naples',              'Naples',               'IT'),
    ('Madrid',              'Madrid',               'ES'),
    ('Barcelona',           'Barcelona',            'ES'),
    ('Warsaw',              'Warsaw',               'PL'),
    ('Krakow',              'Krak%C3%B3w',          'PL'),
    ('Vienna',              'Vienna',               'AT'),
    ('Amsterdam',           'Amsterdam',            'NL'),
    ('Brussels',            'Brussels',             'BE'),
    ('Prague',              'Prague',               'CZ'),
    ('Budapest',            'Budapest',             'HU'),
    ('Bucharest',           'Bucharest',            'RO'),
    ('Stockholm',           'Stockholm',            'SE'),
    ('Oslo',                'Oslo',                 'NO'),
    ('Copenhagen',          'Copenhagen',           'DK'),
    ('Landsberg am Lech',   'Landsberg_am_Lech',    'DE'),
]

print(f'{len(CITIES_META)} cities defined ✅')

## 4. Wikipedia Scraping Functions

In [ ]:
HEADERS  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
BASE_URL = 'https://en.wikipedia.org/wiki/'


def get_soup(slug):
    r = requests.get(BASE_URL + slug, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        print(f'  ⚠️  Wikipedia {r.status_code} for {slug}')
        return None
    return BeautifulSoup(r.content, 'html.parser')


def get_infobox(soup):
    return soup.find('table', class_='infobox')


def get_population(infobox):
    rows = infobox.find_all('tr')
    found = False
    for row in rows:
        text = row.get_text(separator=' ', strip=True)
        if 'Population' in text:
            found = True
            continue
        if found:
            if any(s in text for s in ['Estimate', 'Rank', 'Density', 'Urban', 'Metro']):
                continue
            clean = re.sub(r'\[.*?\]', '', text)
            nums  = re.findall(r'\d[\d,]{4,}', clean)
            for n in nums:
                try:
                    val = int(n.replace(',', ''))
                    if 50000 < val < 20000000:
                        return val
                except Exception:
                    continue
            found = False
    return None


def get_coordinates(soup):
    geo = soup.find('span', class_='geo')
    if geo:
        parts = geo.get_text().split(';')
        if len(parts) == 2:
            try:
                return round(float(parts[0].strip()), 6), round(float(parts[1].strip()), 6)
            except ValueError:
                pass
    return None, None


def decimal_to_dms(decimal_deg, is_lat):
    if decimal_deg is None:
        return None
    direction = ('N' if decimal_deg >= 0 else 'S') if is_lat else ('E' if decimal_deg >= 0 else 'W')
    abs_deg   = abs(decimal_deg)
    degrees   = int(abs_deg)
    minutes   = int((abs_deg - degrees) * 60)
    seconds   = round(((abs_deg - degrees) * 60 - minutes) * 60, 1)
    return f'{degrees}°{minutes}\'{seconds}"{direction}'


print('Wikipedia functions defined ✅')

## 5. Weather Functions (OWM + Open-Meteo)

In [ ]:
def get_owm_forecast(lat, lon, city_name):
    """
    5-day / 3-hour forecast from OWM Free.
    Returns one row per timestamp — matches bootcamp weathers schema:
    forecast_time, outlook, temperature, feels_like, wind_speed, rain_prob
    """
    if lat is None or lon is None:
        return []
    params = {
        'lat':   lat,
        'lon':   lon,
        'appid': OWM_API_KEY,
        'units': 'metric',
        'lang':  'en',
        'cnt':   40
    }
    try:
        r = requests.get('https://api.openweathermap.org/data/2.5/forecast',
                         params=params, timeout=10)
        if r.status_code != 200:
            print(f'  ⚠️  OWM {r.status_code} for {city_name}')
            return []
        rows = []
        for item in r.json()['list']:
            rows.append({
                'forecast_time': item['dt_txt'],
                'outlook':       item['weather'][0]['description'],
                'temperature':   round(item['main']['temp'], 1),
                'feels_like':    round(item['main']['feels_like'], 1),
                'wind_speed':    round(item['wind']['speed'], 1),
                'rain_prob':     round(item.get('pop', 0), 2),
            })
        return rows
    except Exception as e:
        print(f'  ⚠️  OWM error {city_name}: {e}')
        return []


print('Weather functions defined ✅')

## 6. AeroDataBox Functions (Airports + Flights)

In [ ]:
AERO_HEADERS = {
    'x-rapidapi-key':  RAPIDAPI_KEY,
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com'
}


def get_airports(lat, lon, city_name, radius_km=50):
    """
    Search airports near city coordinates using AeroDataBox.
    Returns list of dicts: airport_icao, airport_name
    """
    if lat is None or lon is None:
        return []
    url = f'https://aerodatabox.p.rapidapi.com/airports/search/location/{lat}/{lon}/km/{radius_km}/16'
    params = {'withFlightInfoOnly': 'true'}
    try:
        r = requests.get(url, headers=AERO_HEADERS, params=params, timeout=10)
        if r.status_code != 200:
            print(f'  ⚠️  AeroDataBox airports {r.status_code} for {city_name}')
            return []
        items = r.json().get('items', [])
        airports = []
        for item in items:
            airports.append({
                'airport_icao': item.get('icao', ''),
                'airport_name': item.get('name', ''),
            })
        return airports
    except Exception as e:
        print(f'  ⚠️  AeroDataBox airports error {city_name}: {e}')
        return []


def get_flights(icao, city_name):
    """
    Get tomorrow's arrivals for a given airport ICAO code.
    Returns list of dicts: flight_num, departure_icao, arrival_icao, arrival_time
    """
    tomorrow    = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
    time_from   = f'{tomorrow}T00:00'
    time_to     = f'{tomorrow}T23:59'
    url         = f'https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{time_from}/{time_to}'
    params      = {
        'withLeg':          'true',
        'direction':        'Arrival',
        'withCancelled':    'false',
        'withCodeshared':   'false',
        'withCargo':        'false',
        'withPrivate':      'false',
    }
    try:
        r = requests.get(url, headers=AERO_HEADERS, params=params, timeout=10)
        if r.status_code != 200:
            print(f'  ⚠️  Flights {r.status_code} for {icao}')
            return []
        arrivals = r.json().get('arrivals', [])
        flights  = []
        for flight in arrivals:
            try:
                flights.append({
                    'flight_num':     flight.get('number', ''),
                    'departure_icao': flight.get('departure', {}).get('airport', {}).get('icao', ''),
                    'arrival_icao':   icao,
                    'arrival_time':   flight.get('arrival', {}).get('scheduledTime', {}).get('local', ''),
                })
            except Exception:
                continue
        return flights
    except Exception as e:
        print(f'  ⚠️  Flights error {icao}: {e}')
        return []


print('AeroDataBox functions defined ✅')

## 7. Scrape All Cities (Wikipedia)

In [ ]:
cities_rows  = []
city_id      = 1

for city_name, wiki_slug, country_code in CITIES_META:
    print(f'Scraping {city_name}...', end=' ')
    try:
        soup    = get_soup(wiki_slug)
        if soup is None:
            print('SKIP')
            continue
        infobox = get_infobox(soup)
        if infobox is None:
            print('SKIP (no infobox)')
            continue

        lat, lon = get_coordinates(soup)
        pop      = get_population(infobox)

        cities_rows.append({
            'city_id':      city_id,
            'city_name':    city_name,
            'country_code': country_code,
            'latitude':     lat,
            'longitude':    lon,
            'lat_dms':      decimal_to_dms(lat, is_lat=True),
            'lon_dms':      decimal_to_dms(lon, is_lat=False),
            '_population':  pop,   # temp — goes to populations table
        })

        pop_str = f'{pop:,}' if pop else 'N/A'
        print(f'✅  lat={lat}  lon={lon}  pop={pop_str}')
        city_id += 1

    except Exception as e:
        print(f'❌ {e}')
    time.sleep(1)

print(f'\n{len(cities_rows)} cities scraped ✅')

## 8. Fix Missing Populations

In [ ]:
manual_pop = {
    'Paris':              2102650,
    'Lyon':                522228,
    'Marseille':           873076,
    'Landsberg am Lech':   32000,
}

for city in cities_rows:
    if city['_population'] is None and city['city_name'] in manual_pop:
        city['_population'] = manual_pop[city['city_name']]
        print(f"Fixed {city['city_name']}: {manual_pop[city['city_name']]:,}")

# Check remaining NaN
missing = [c['city_name'] for c in cities_rows if c['_population'] is None]
if missing:
    print(f'\n⚠️  Still missing: {missing}')
else:
    print('\nAll populations complete ✅')

## 9. Fetch Weather for All Cities (OWM)

In [ ]:
weather_rows = []
weather_id   = 1

for city in cities_rows:
    name = city['city_name']
    lat  = city['latitude']
    lon  = city['longitude']
    cid  = city['city_id']

    print(f'Weather {name}...', end=' ')
    forecasts = get_owm_forecast(lat, lon, name)

    for row in forecasts:
        weather_rows.append({
            'id':            weather_id,
            'city_id':       cid,
            'forecast_time': row['forecast_time'],
            'outlook':       row['outlook'],
            'temperature':   row['temperature'],
            'feels_like':    row['feels_like'],
            'wind_speed':    row['wind_speed'],
            'rain_prob':     row['rain_prob'],
        })
        weather_id += 1

    print(f'✅  {len(forecasts)} timestamps')
    time.sleep(0.5)

print(f'\nWeather done! {len(weather_rows)} total rows ✅')

## 10. Fetch Airports + Flights (AeroDataBox)

In [ ]:
airports_rows       = []   # airports table — unique per ICAO
cities_airports_rows = []  # junction table city_id ↔ airport_icao
flights_rows        = []
flight_id           = 1
seen_icao           = set()

for city in cities_rows:
    name = city['city_name']
    lat  = city['latitude']
    lon  = city['longitude']
    cid  = city['city_id']

    print(f'Airports {name}...', end=' ')
    airports = get_airports(lat, lon, name)

    if not airports:
        print('none found')
        time.sleep(0.5)
        continue

    for ap in airports:
        icao = ap['airport_icao']
        if not icao:
            continue

        # airports table — unique
        if icao not in seen_icao:
            airports_rows.append({
                'airport_icao': icao,
                'airport_name': ap['airport_name'],
            })
            seen_icao.add(icao)

        # cities_airports junction
        cities_airports_rows.append({
            'city_id':      cid,
            'airport_icao': icao,
        })

        # flights for this airport
        print(f'\n  Flights {icao}...', end=' ')
        flights = get_flights(icao, name)
        for fl in flights:
            flights_rows.append({
                'flight_id':      flight_id,
                'flight_num':     fl['flight_num'],
                'departure_icao': fl['departure_icao'],
                'arrival_icao':   fl['arrival_icao'],
                'arrival_time':   fl['arrival_time'],
            })
            flight_id += 1
        print(f'{len(flights)} flights')
        time.sleep(1)

    time.sleep(0.5)

print(f'\nAirports: {len(airports_rows)}  |  cities_airports: {len(cities_airports_rows)}  |  flights: {len(flights_rows)} ✅')

## 11. Build All DataFrames

In [ ]:
# cities
cities_df = pd.DataFrame(cities_rows)[[
    'city_id', 'city_name', 'country_code',
    'latitude', 'longitude', 'lat_dms', 'lon_dms'
]]

# populations
populations_df = pd.DataFrame([{
    'city_id':              c['city_id'],
    'population':           c['_population'],
    'timestamp_population': datetime.now().year,
} for c in cities_rows])

# weathers
weathers_df = pd.DataFrame(weather_rows)

# airports
airports_df = pd.DataFrame(airports_rows) if airports_rows else pd.DataFrame(columns=['airport_icao', 'airport_name'])

# cities_airports
cities_airports_df = pd.DataFrame(cities_airports_rows) if cities_airports_rows else pd.DataFrame(columns=['city_id', 'airport_icao'])

# flights
flights_df = pd.DataFrame(flights_rows) if flights_rows else pd.DataFrame(columns=['flight_id', 'flight_num', 'departure_icao', 'arrival_icao', 'arrival_time'])

print(f'cities:          {len(cities_df)} rows')
print(f'populations:     {len(populations_df)} rows')
print(f'weathers:        {len(weathers_df)} rows')
print(f'airports:        {len(airports_df)} rows')
print(f'cities_airports: {len(cities_airports_df)} rows')
print(f'flights:         {len(flights_df)} rows')

In [ ]:
# Quick previews
print('CITIES:'); display(cities_df.head(3))
print('WEATHERS:'); display(weathers_df.head(3))
print('AIRPORTS:'); display(airports_df.head(3))
print('FLIGHTS:'); display(flights_df.head(3))

## 12. Export to MySQL

Schema exactă din bootcamp diagram.

In [ ]:
conn_str     = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}'
engine_no_db = create_engine(conn_str)

try:
    engine.dispose()
except Exception:
    pass

with engine_no_db.connect() as conn:
    conn.execute(text(f'DROP DATABASE IF EXISTS `{DB_NAME}`'))
    conn.execute(text(f'CREATE DATABASE `{DB_NAME}` CHARACTER SET utf8mb4'))
    conn.commit()

engine = create_engine(f'{conn_str}/{DB_NAME}')
print(f'Database `{DB_NAME}` created ✅')

In [ ]:
with engine.connect() as conn:

    # 1. cities
    conn.execute(text('''
        CREATE TABLE cities (
            city_id      INT           PRIMARY KEY AUTO_INCREMENT,
            city_name    VARCHAR(200)  NOT NULL,
            country_code VARCHAR(3),
            latitude     DECIMAL(9,6),
            longitude    DECIMAL(9,6),
            lat_dms      VARCHAR(25),
            lon_dms      VARCHAR(25)
        )
    '''))

    # 2. populations
    conn.execute(text('''
        CREATE TABLE populations (
            city_id              INT  NOT NULL,
            population           INT,
            timestamp_population YEAR,
            FOREIGN KEY (city_id) REFERENCES cities(city_id)
        )
    '''))

    # 3. weathers
    conn.execute(text('''
        CREATE TABLE weathers (
            id            INT           PRIMARY KEY AUTO_INCREMENT,
            city_id       INT           NOT NULL,
            forecast_time DATETIME,
            outlook       VARCHAR(200),
            temperature   DECIMAL(5,2),
            feels_like    DECIMAL(5,0),
            wind_speed    DECIMAL(5,2),
            rain_prob     DECIMAL(5,2),
            FOREIGN KEY (city_id) REFERENCES cities(city_id)
        )
    '''))

    # 4. airports
    conn.execute(text('''
        CREATE TABLE airports (
            airport_icao VARCHAR(25)  PRIMARY KEY,
            airport_name VARCHAR(255)
        )
    '''))

    # 5. cities_airports (junction)
    conn.execute(text('''
        CREATE TABLE cities_airports (
            city_id      INT         NOT NULL,
            airport_icao VARCHAR(25) NOT NULL,
            FOREIGN KEY (city_id)      REFERENCES cities(city_id),
            FOREIGN KEY (airport_icao) REFERENCES airports(airport_icao)
        )
    '''))

    # 6. flights
    conn.execute(text('''
        CREATE TABLE flights (
            flight_id      INT          PRIMARY KEY AUTO_INCREMENT,
            flight_num     VARCHAR(25),
            departure_icao VARCHAR(25),
            arrival_icao   VARCHAR(25),
            arrival_time   DATETIME,
            FOREIGN KEY (arrival_icao) REFERENCES airports(airport_icao)
        )
    '''))

    conn.commit()

print('6 tables created ✅')

In [ ]:
# Load in FK order
cities_df.to_sql('cities',           engine, if_exists='append', index=False)
populations_df.to_sql('populations', engine, if_exists='append', index=False)
weathers_df.to_sql('weathers',       engine, if_exists='append', index=False)

if len(airports_df) > 0:
    airports_df.to_sql('airports',             engine, if_exists='append', index=False)
    cities_airports_df.to_sql('cities_airports', engine, if_exists='append', index=False)

if len(flights_df) > 0:
    flights_df.to_sql('flights', engine, if_exists='append', index=False)

print('All data loaded into MySQL ✅')

## 13. Verify with SQL Queries

In [ ]:
for table in ['cities', 'populations', 'weathers', 'airports', 'cities_airports', 'flights']:
    df = pd.read_sql(f'SELECT COUNT(*) as `row_count` FROM `{table}`', engine)
    print(f'{table:20s} → {df["row_count"][0]} rows')

In [ ]:
# Cities + current weather
query = '''
    SELECT
        c.city_name,
        c.country_code,
        c.lat_dms,
        c.lon_dms,
        p.population,
        w.forecast_time,
        w.temperature,
        w.outlook,
        w.wind_speed,
        w.rain_prob
    FROM cities c
    JOIN populations p ON c.city_id = p.city_id
    JOIN weathers    w ON c.city_id = w.city_id
    WHERE w.forecast_time = (
        SELECT MIN(forecast_time) FROM weathers w2
        WHERE w2.city_id = c.city_id
    )
    ORDER BY p.population DESC
    LIMIT 10
'''
pd.read_sql(query, engine)

In [ ]:
# Airports per city
query = '''
    SELECT
        c.city_name,
        a.airport_icao,
        a.airport_name
    FROM cities_airports ca
    JOIN cities   c ON ca.city_id      = c.city_id
    JOIN airports a ON ca.airport_icao = a.airport_icao
    ORDER BY c.city_name
'''
pd.read_sql(query, engine)

In [ ]:
# Tomorrow's arrivals — full picture
query = '''
    SELECT
        f.flight_num,
        f.departure_icao,
        f.arrival_icao,
        f.arrival_time,
        a.airport_name,
        c.city_name
    FROM flights f
    JOIN airports        a  ON f.arrival_icao  = a.airport_icao
    JOIN cities_airports ca ON a.airport_icao  = ca.airport_icao
    JOIN cities          c  ON ca.city_id      = c.city_id
    ORDER BY f.arrival_time
    LIMIT 20
'''
pd.read_sql(query, engine)

## 14. Challenge 🔥

1. **Which city has the most airports within 50km?**
2. **Show tomorrow's flights arriving in Berlin, sorted by arrival time.**
3. **Which city has the highest rain probability in the next 24h?**
4. **Find all airports that receive flights from outside Europe** *(departure_icao NOT starting with E)*
5. **BONUS**: Which city has the best weather tomorrow? *(highest temp + lowest rain_prob)*